In [ ]:
!pip install keras==2.10.0
1.26.4

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!rsync -a --info=progress2 "/content/drive/MyDrive/COCO_W_KAAKAA_dataset/dataset.zip" "/content"

  9,267,846,144 100%   50.68MB/s    0:02:54 (xfr#1, to-chk=0/1)


In [ ]:
import zipfile
with zipfile.ZipFile("dataset.zip", 'r') as zip_ref:
    zip_ref.extractall("./dataset/")

In [ ]:
!pip install ultralytics
%cd ./dataset

ImportError: cannot import name 'ImageDataGenerator' from 'keras.preprocessing.image' (/usr/local/lib/python3.11/dist-packages/keras/preprocessing/image/__init__.py)

In [ ]:
#Forgot to update the class labels on my end so this script does that here
#80 is the kaakaa class
from pathlib import Path

label_dirs = [Path("labels/train"), Path("labels/val")]
updated_count = 0

for labels_dir in label_dirs:
    for txt_file in labels_dir.rglob("*.txt"):
        # Skip files starting with "0000"
        if txt_file.stem.startswith("0000"):
            continue

        with open(txt_file, "r") as f:
            lines = f.readlines()

        changed = False
        new_lines = []
        for line in lines:
            parts = line.strip().split()
            if parts and parts[0] == '0':
                parts[0] = '80'
                changed = True
            new_lines.append(" ".join(parts))

        if changed:
            updated_count += 1
            with open(txt_file, "w") as f:
                f.write("\n".join(new_lines) + "\n")

print(f"✅ Updated class IDs from 0 to 80 in {updated_count} label files.")

In [ ]:
import os

# Paths
images_dirs = ["images/train", "images/val"]
labels_dirs = ["labels/train", "labels/val"]

for images_dir, labels_dir in zip(images_dirs, labels_dirs):
    missing_labels = []

    for img_file in os.listdir(images_dir):
        if not img_file.lower().endswith((".jpg", ".jpeg", ".png")):
            continue
        if img_file.startswith("0000"):
            continue

        base_name = os.path.splitext(img_file)[0]
        label_path = os.path.join(labels_dir, base_name + ".txt")

        if not os.path.exists(label_path):
            missing_labels.append(img_file)

    print(f"\nChecking {images_dir}:")
    if missing_labels:
        print(f"  Missing labels for {len(missing_labels)} images:")
        for f in missing_labels:
            print(f"    {f}")
    else:
        print("  ✅ All non-0000 images have matching labels.")

In [ ]:
#making sure bird and kaakaa classes are equal...
import os
import random

# Paths
labels_dirs = ["labels/train", "labels/val"]
images_dirs = ["images/train", "images/val"]

# Classes
bird_class = "14"
kaakaa_class = "80"

# Fraction of pure negatives to delete
negative_delete_fraction = 0.97

def process_set(labels_dir, images_dir):
    label_files = [f for f in os.listdir(labels_dir) if f.endswith(".txt")]
    random.shuffle(label_files)

    birds = []
    kaakaas = []
    others = []

    # Categorize images
    for lf in label_files:
        label_path = os.path.join(labels_dir, lf)
        with open(label_path, "r") as f:
            contents = f.read().strip()

        labels = [line.split()[0] for line in contents.splitlines() if line.strip()]

        if bird_class in labels and kaakaa_class not in labels:
            birds.append(lf)
        elif kaakaa_class in labels and bird_class not in labels:
            kaakaas.append(lf)
        elif bird_class not in labels and kaakaa_class not in labels:
            others.append(lf)

    # Step 1: Match Bird count to Kaakaa count
    kaakaa_count = len(kaakaas)
    bird_count = len(birds)
    birds_to_delete = set()

    if bird_count > kaakaa_count:
        delete_count = bird_count - kaakaa_count
        birds_to_delete = set(random.sample(birds, delete_count))

        for lf in birds_to_delete:
            os.remove(os.path.join(labels_dir, lf))
            base_name = os.path.splitext(lf)[0]
            for ext in [".jpg", ".jpeg", ".png"]:
                img_path = os.path.join(images_dir, base_name + ext)
                if os.path.exists(img_path):
                    os.remove(img_path)
                    break
        print(f"Deleted {delete_count} extra Bird files to match Kaakaa count.")

    # Step 2: Delete 97% of pure negatives
    delete_count_neg = int(len(others) * negative_delete_fraction)
    negs_to_delete = set(random.sample(others, delete_count_neg))

    for lf in negs_to_delete:
        os.remove(os.path.join(labels_dir, lf))
        base_name = os.path.splitext(lf)[0]
        for ext in [".jpg", ".jpeg", ".png"]:
            img_path = os.path.join(images_dir, base_name + ext)
            if os.path.exists(img_path):
                os.remove(img_path)
                break

    print(f"Processed {labels_dir}:")
    print(f"  Birds kept: {len(birds) - len(birds_to_delete)}")
    print(f"  Kaakaas kept: {kaakaa_count}")
    print(f"  Negatives kept: {len(others) - delete_count_neg}")
    print(f"  Negatives deleted: {delete_count_neg}")

# Run for both train and val sets
for labels_dir, images_dir in zip(labels_dirs, images_dirs):
    process_set(labels_dir, images_dir)

In [ ]:
#Prints number of files in val and train
import os
import shutil

def list_size(directory):
  absolute_path = os.path.abspath(os.path.realpath(directory))
  print(f"Resolved absolute path: {absolute_path}")
  file_count = sum(len(files) for _, _, files in os.walk(directory))
  print(f"Number of files in {directory}: {file_count}")

list_size("images/train")
list_size("images/val")

In [ ]:
!yolo settings runs_dir="/content/drive/MyDrive/yolo_runs"

In [ ]:
!yolo train model="/content/drive/MyDrive/yolo_runs/segment/train15/weights/best.pt" data=kaakaa_data.yaml epochs=150 imgsz=640 classes=[80]

In [ ]:
"""function ClickConnect(){
    console.log("Clicking");
    document.querySelector("colab-toolbar-button#toolbar-show-command-palette").click();
    document.querySelector("colab-toolbar-button.inputarea.horizontal.layout.code editor.flex.monaco").click();
}
setInterval(ClickConnect, 60000);
#put this in the console to avoid timeouts
I think colab no longer works like this circa 2022 but if I don't believe I've done something to prevent a timeout I will sit here watching epochs pass for hours
placebo effect is still an effect"""